# fase_2 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v6_example
Connected to new database: 1


## Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()

# Mengambil nama tabel dari hasil query
# Note: Format output 'SHOW TABLES' biasanya {'Tables_in_dbname': 'tablename'}
target_tables = [list(t.values())[0] for t in tables_data]

print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")
print(target_tables)

# Dictionary untuk menyimpan data yang sudah di-load
df_old = {}

print("\n--- Memulai proses load semua data tabel ---")

for table in target_tables:
    try:
        # Load data menggunakan pandas langsung dari koneksi SQL
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")
print("Kamu sekarang bisa akses datanya dengan: df_old['nama_tabel']")

# Contoh akses data:
# print(df_old['users'].head())

# Tutup koneksi jika sudah tidak digunakan
# db_old.close()
# db_new.close()


--- Ditemukan 115 tabel di Database Lama ---
['absensi', 'absensi_note', 'bidang', 'bidangkategori', 'bidanglink', 'cache', 'cache_locks', 'calon', 'calon_detil', 'calon_pertanyaan', 'calon_pertanyaan_detil', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'catatan_siswa_follow_up', 'catatanawal_admin', 'catatanawal_datautama', 'catatanawal_infolain', 'catatanawal_tglpenting', 'divisi', 'docs', 'failed_jobs', 'file_rapor_siswa', 'form', 'form_calon', 'form_calon_detil1', 'form_calon_detil2', 'form_calon_detil3', 'form_calon_detil4', 'format_rapor', 'format_rapor_detil', 'format_rapor_detil_rumus', 'format_rapor_rumus', 'format_raport_level', 'hakakses', 'histori_pengajuan', 'history_rapor', 'identitas', 'infrastruktur', 'jabatan', 'jadwal', 'jadwal_detil', 'jadwal_pengajar', 'jadwal_siswa', 'jamkerja', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'kecamatan', 'keluar', 'keluarga', 'kelurahan', 'kurikulum', 'kurikulum_detil', 'kurikulum_detil_sub', 'kuriku

In [4]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS (DATABASE BARU)
# ---------------------------------------------------------
# Menggunakan cursor dari database baru
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()

# Mengambil nama tabel dari hasil query
target_tables_new = [list(t.values())[0] for t in tables_data_new]

print(f"\n--- Ditemukan {len(target_tables_new)} tabel di Database Baru ---")
print(target_tables_new)

# Dictionary untuk menyimpan data dari database baru (jika diperlukan untuk verifikasi)
df_new = {}

print("\n--- Memulai proses load semua data dari Database Baru ---")

for table in target_tables_new:
    try:
        # Load data menggunakan pandas dengan koneksi database baru
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table} dari DB Baru: {e}")

print("\n--- Proses load selesai. Data DB Baru tersimpan di 'df_new' ---")


--- Ditemukan 113 tabel di Database Baru ---
['absensi', 'activity_log', 'admin_sarpras', 'bidang_kategori', 'bidang_link', 'busdev_bidang', 'cache', 'cache_locks', 'calon_siswa', 'calon_siswa_akademik', 'calon_siswa_bayar', 'calon_siswa_fo_detail', 'calon_siswa_form_program_requirements', 'calon_siswa_form_programs', 'calon_siswa_jadwal', 'calon_siswa_kursus', 'calon_siswa_ortu', 'calon_siswa_proses', 'calon_siswa_proses_logs', 'calon_siswa_status_logs', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_remidi_siswa', 'catatan_siswa', 'division_user', 'divisions', 'failed_jobs', 'followup_cs', 'histori_pengajuan', 'izin_karyawan', 'jadwal', 'jadwal_detail', 'jadwal_detail_logs', 'jadwal_hari', 'jadwal_pengajar', 'jadwal_siswa', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'karyawan_resign', 'kecamatan', 'keluarga_karyawan', 'kelurahan', 'kemitraan_verifikator', 'kontak_prospek', 'kursus', 'kursus_level', 'kursus_libur', 'kursus_siswa', 'level', 'libur', 'log_aktiv

# Fixing Tabel Kariawan

## Merge df_old[users] dan df_new[users]

karyawan, keluarga_karyawan, bidang_kategori, bidang_link.

In [5]:
id_karyawan_lama = df_old['karyawan']['idkaryawan'].astype(str).str.strip()

In [6]:
df_old['users'].head()

,idusers,email,pass,nama,foto,idrole,wa,thnbekerja,idjabatan,idjamkerja,...,ispurchase,isteaching,ishr,isga,isit,ispdd,isbusdev,ispimpinan,ttd,expertise
0,U00001,ditari@leapsurabaya.sch.id,aGtq,ADMINISTRATOR,logo.png,R00001,0851-7438-7539,2023-06-02,J00005,2.0,...,0.0,0.0,0,1,0,0,0,0,None,None
1,U00003,graciela@leapsurabaya.sch.id,qWmlbcVjYmo%3D,"Graciela Evanda Ronadi, S.Kom.",1707279559_ec864cc58f50e8b890d5.jpg,R00003,0812-3447-7137,2023-03-13,J00006,2.0,...,0.0,1.0,0,0,0,0,0,0,1702002184_d82747bcbd0f2bdab0a7.png,None
2,U00011,daniar.rizki@leapsurabaya.sch.id,aGtq,DANIAR AULIA RIZKI,1692700456_2d6352eb557d734e53b7.jpeg,R00006,0896-9632-0278,2020-12-01,J00010,2.0,...,0.0,0.0,0,0,0,0,1,0,None,
3,U00012,habibah.elfiani@leapsurabaya.sch.id,o56YqZJkZA%3D%3D,Habibah Melyna,1686127161_1ec3d11da554fb668e7c.jpg,R00003,0857-3093-3317,2020-03-02,J00002,3.0,...,0.0,1.0,0,0,0,1,1,0,1702892302_89507e3f2d87fb0dfdee.png,desain grafis
4,U00014,laksmi.p@leapsurabaya.sch.id,aWlpbw%3D%3D,Laksmi Puspitowardhani,1696338938_2a535db26e3f89919325.jpg,R00008,0821-3996-6817,2018-07-31,J00004,3.0,...,0.0,0.0,0,0,0,1,0,0,None,


In [7]:
df_new['users'].head()

,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,2026-06-23 21:40:16,qWmlbcVjYmo%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16
3,U00012,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id,2026-06-23 21:40:16,o56YqZJkZA%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16
4,U00014,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id,2026-06-23 21:40:16,aWlpbw%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16


In [8]:
df_old['karyawan'].head()

,idkaryawan,ktp,nickname,kota,tgl,jk,goldar,agama,status,alamatktp,...,bpjskerja,bpjssehat,idusers,nama,anak,rekening,moda,ig,fb,link
0,LEAP001VI02,None,None,None,None,Wanita,None,None,None,None,...,None,None,U00001,None,None,None,None,None,None,
1,LEAP003III2023,3514186411980002,Graciela,Sidoarjo,11/24/1998,Wanita,-,Islam,Belum Kawin,"Ranggeh, Gondangwetan, Pasuruan",...,,,U00003,Graciela Evanda Ronadi,,0140881385821,Motor Pribadi,https://www.instagram.com/gracielaevr/,,https://drive.google.com/drive/folders/1alw9Su...
2,LEAP011XII01,3515135105910001,DANIAR,SURABAYA,05/11/1991,Wanita,B,ISLAM,KAWIN,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSO...",...,21034278636,0002319622514,U00011,DANIAR AULIA RIZKI,1,1410022279715,SEPEDA MOTOR,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va...
3,LEAP012III02,3578106705930001,Habibah,Semarang,05/27/1993,Wanita,AB,Islam,Belum menikah,Jl Pacar Kembang Vc No 22,...,21034278651,0002999218792,U00012,Habibah Melyna Elfiani,,1420018314442,Sepeda Motor Pribadi,https://instagram.com/habibahmelyna?igshid=MzN...,,None
4,LEAP014VII31,3578035706820005,Laksmi,Surabaya,06/17/1982,Wanita,O,ISLAM,Single,Rungkut Asri Barat XIII no 17,...,18088838356,00015384308,U00014,Laksmi Puspitowardhani,,1420016807967,Motor,https://www.instagram.com/laksmi_purplespace/?...,-,https://drive.google.com/drive/folders/1-7iI4-...


In [9]:
# Gabungin df_old['users'] dengan df_new['users'] berdasarkan ID, tapi hanya untuk kolom yang ada di df_old dan tidak ada di df_new
cols_to_keep_from_old = [col for col in df_old['users'].columns 
                        if col not in ['nama', 'pass'] 
                        and col not in df_new['users'].columns]

# Merge berdasarkan id_user (new) dan idusers (old)
df_merged_users = df_new['users'].merge(
    df_old['users'][['idusers'] + [c for c in cols_to_keep_from_old if c != 'idusers']],
    left_on='id_user',
    right_on='idusers',
    how='left'
)

df_merged_users.head()

,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at,idusers,foto,...,ispurchase,isteaching,ishr,isga,isit,ispdd,isbusdev,ispimpinan,ttd,expertise
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00001,logo.png,...,0.0,0.0,0,1,0,0,0,0,None,None
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,2026-06-23 21:40:16,qWmlbcVjYmo%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00003,1707279559_ec864cc58f50e8b890d5.jpg,...,0.0,1.0,0,0,0,0,0,0,1702002184_d82747bcbd0f2bdab0a7.png,None
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00011,1692700456_2d6352eb557d734e53b7.jpeg,...,0.0,0.0,0,0,0,0,1,0,None,
3,U00012,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id,2026-06-23 21:40:16,o56YqZJkZA%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00012,1686127161_1ec3d11da554fb668e7c.jpg,...,0.0,1.0,0,0,0,1,1,0,1702892302_89507e3f2d87fb0dfdee.png,desain grafis
4,U00014,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id,2026-06-23 21:40:16,aWlpbw%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00014,1696338938_2a535db26e3f89919325.jpg,...,0.0,0.0,0,0,0,1,0,0,None,


In [10]:
# Analisis df_merged_users untuk duplicate columns, null values, dan informasi lainnya

print("="*80)
print("ANALISIS df_merged_users")
print("="*80)

# 1. Info umum dataframe
print("\n1. INFORMASI UMUM:")
print(f"   Shape: {df_merged_users.shape}")
print(f"   Total Rows: {len(df_merged_users)}")
print(f"   Total Columns: {len(df_merged_users.columns)}")

# 2. Cek duplicate columns
print("\n2. DUPLICATE COLUMNS:")
duplicate_cols = df_merged_users.columns[df_merged_users.columns.duplicated()]
if len(duplicate_cols) > 0:
    print(f"   ⚠️  Ditemukan {len(duplicate_cols)} duplicate column(s):")
    for col in duplicate_cols.unique():
        print(f"      - {col}")
else:
    print("   ✓ Tidak ada duplicate columns")

# 3. Cek null values per column
print("\n3. NULL VALUES PER COLUMN:")
null_counts = df_merged_users.isnull().sum()
null_percentage = (null_counts / len(df_merged_users)) * 100
null_info = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Percentage': null_percentage.values
}).sort_values('Null_Count', ascending=False)

null_info_display = null_info[null_info['Null_Count'] > 0]
if len(null_info_display) > 0:
    print(null_info_display.to_string(index=False))
else:
    print("   ✓ Tidak ada null values")

# 4. Data types
print("\n4. DATA TYPES:")
# Display data types with a more detailed summary
dtype_summary = pd.DataFrame({
    'Column': df_merged_users.dtypes.index,
    'Data_Type': df_merged_users.dtypes.values
})
print(dtype_summary.to_string(index=False))


print("\n5. VALUE COUNTS PER COLUMN:")
print("=" * 80)

for col in df_merged_users.columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df_merged_users[col].nunique()}")
    if df_merged_users[col].dtype in ['object', 'str', 'int64', 'float64']:
        # Untuk kolom string/object, tampilkan value counts
        value_counts = df_merged_users[col].value_counts(dropna=False)
        if len(value_counts) <= 10:
            print(value_counts)
        else:
            print(value_counts.head(10))
            print(f"  ... dan {len(value_counts) - 10} nilai lainnya")
    else:
        # Untuk kolom numerik, tampilkan statistik
        print(f"  Min: {df_merged_users[col].min()}")
        print(f"  Max: {df_merged_users[col].max()}")
        print(f"  Mean: {df_merged_users[col].mean():.2f}")

ANALISIS df_merged_users

1. INFORMASI UMUM:
   Shape: (52, 28)
   Total Rows: 52
   Total Columns: 28

2. DUPLICATE COLUMNS:
   ✓ Tidak ada duplicate columns

3. NULL VALUES PER COLUMN:
        Column  Null_Count  Null_Percentage
remember_token          52       100.000000
      idbidang          41        78.846154
    idjamkerja          38        73.076923
           ttd          37        71.153846
     expertise          11        21.153846
          foto           8        15.384615
         minat           1         1.923077
    isteaching           1         1.923077
            wa           1         1.923077
    ispurchase           1         1.923077

4. DATA TYPES:
           Column      Data_Type
          id_user         object
             name         object
            email         object
email_verified_at datetime64[ns]
         password         object
   remember_token         object
       created_at datetime64[ns]
       updated_at datetime64[ns]
          iduser

## Benerin nama dan nickname

nama_lengkap di tabel kariawan tidak digunakan.

In [11]:
# ---------------------------------------------------------
# CEK KESAMAAN NAMA: df_old['karyawan'] vs df_new['users']
# ---------------------------------------------------------

# 1. Ambil kolom yang diperlukan
old_karyawan = df_old['karyawan'][['idusers', 'nama']].copy()
new_users = df_new['users'][['id_user', 'name']].copy()

# 2. Gabungkan data berdasarkan acuan ID
comparison = pd.merge(
    old_karyawan, 
    new_users, 
    left_on='idusers', 
    right_on='id_user', 
    how='inner'
)

# 3. Bandingkan kolom nama dan name (gunakan strip untuk akurasi)
comparison['is_same'] = comparison['nama'].str.strip() == comparison['name'].str.strip()

# 4. Tampilkan ringkasan hasil
total_data = len(comparison)
total_sama = comparison['is_same'].sum()
total_beda = total_data - total_sama

print(f"Total data yang dicek: {total_data}")
print(f"Data yang sama: {total_sama}")
print(f"Data yang berbeda: {total_beda}")

# 5. Tampilkan daftar yang berbeda jika ada
if total_beda > 0:
    print("\n--- Daftar Perbedaan (Mismatches) ---")
    mismatches = comparison[comparison['is_same'] == False]
    display(mismatches[['id_user', 'nama', 'name']])
else:
    print("\n✅ Semua nama sudah sesuai antara df_old['karyawan'] dan df_new['users']!")

Total data yang dicek: 52
Data yang sama: 22
Data yang berbeda: 30

--- Daftar Perbedaan (Mismatches) ---


,id_user,nama,name
0,U00001,None,ADMINISTRATOR
1,U00003,Graciela Evanda Ronadi,"Graciela Evanda Ronadi, S.Kom."
3,U00012,Habibah Melyna Elfiani,Habibah Melyna
5,U00015,None,Ika Asriani Yadin
6,U00016,Luluk Fatikah Sari,"Luluk Fatikah Sari, S.Pd."
7,U00018,Ditari Kurnia Damayanti,Ditari Kurnia
8,U00019,Hartatik,"Hartatik, S.S."
10,U00023,Titi Hapsari Retnaningtias,Titi Hapsari
11,U00026,Qorin Rahmaniyah,"Qorin Rahmaniyah, S.Pd."
12,U00027,fgz,Test


In [12]:
# ---------------------------------------------------------
# BANDINGKAN NAMA (BARU) DENGAN NICKNAME (LAMA)
# ---------------------------------------------------------

comparison_nickname = pd.merge(
    df_old['karyawan'][['idusers', 'nickname']].copy(),
    df_new['users'][['id_user', 'name']].copy(),
    left_on='idusers',
    right_on='id_user',
    how='inner'
)

print("--- Hasil Perbandingan: Users Name vs Old Nickname ---")
display(comparison_nickname[['id_user', 'name', 'nickname']])

--- Hasil Perbandingan: Users Name vs Old Nickname ---


,id_user,name,nickname
0,U00001,ADMINISTRATOR,None
1,U00003,"Graciela Evanda Ronadi, S.Kom.",Graciela
2,U00011,DANIAR AULIA RIZKI,DANIAR
3,U00012,Habibah Melyna,Habibah
4,U00014,Laksmi Puspitowardhani,Laksmi
5,U00015,Ika Asriani Yadin,None
6,U00016,"Luluk Fatikah Sari, S.Pd.",Luluk
7,U00018,Ditari Kurnia,Tari
8,U00019,"Hartatik, S.S.",Tatik
9,U00020,Juni Arlianto,MJ


In [13]:
# ---------------------------------------------------------
# 1. IDENTIFIKASI NICKNAME BERMASALAH (NULL ATAU NO. HP)
# ---------------------------------------------------------
import re

def is_phone_number(val):
    if pd.isna(val) or val is None:
        return True # Anggap bermasalah jika null
    return bool(re.search(r'[0-9]{8,}', str(val)))

# Merge dulu agar bisa membandingkan Name (Baru) dan Nickname (Lama)
df_nick_check = pd.merge(
    df_old['karyawan'][['idusers', 'nickname']].copy(),
    df_new['users'][['id_user', 'name']].copy(),
    left_on='idusers',
    right_on='id_user',
    how='inner'
)

# Filter yang bermasalah
mask_problem = df_nick_check['nickname'].apply(is_phone_number)
problem_nicknames = df_nick_check[mask_problem].copy()

print(f"Ditemukan {len(problem_nicknames)} nickname bermasalah.\n")
if not problem_nicknames.empty:
    print("--- Tabel Pengecekan: Name (Baru) vs Nickname (Bermasalah) ---")
    display(problem_nicknames[['id_user', 'name', 'nickname']])
else:
    print("✅ Tidak ada nickname yang NULL atau berupa Nomor HP.")

Ditemukan 7 nickname bermasalah.

--- Tabel Pengecekan: Name (Baru) vs Nickname (Bermasalah) ---


,id_user,name,nickname
0,U00001,ADMINISTRATOR,None
5,U00015,Ika Asriani Yadin,None
13,U00028,pengajar,None
23,U00040,"Agung Wijayanto, S.T.",083115270222
28,U00046,sosialmedia1,None
30,U00048,"Yerly A Datu, S.Pd, M.Pd, CT.ACT, C.PS",None
34,U00053,Ann Clariss Agbalog Amora,None


In [14]:
# ---------------------------------------------------------
# 3. INPUT NICKNAME MANUAL UNTUK ID SPESIFIK
# ---------------------------------------------------------
# Silakan isi nama di dalam tanda kutip '', lalu jalankan sel ini.

nickname_fixes = {
    'U00001': 'ADMINISTRATOR',  # Isi nickname baru di sini
    'U00015': 'Ika',
    'U00028': 'pengajar',
    'U00040': 'Agung',
    'U00046': 'sosmed',
    'U00048': 'Yerly',
    'U00053': 'Ann'
}

for id_user, new_nick in nickname_fixes.items():
    if new_nick.strip(): # Hanya update jika tidak kosong
        df_old['karyawan'].loc[df_old['karyawan']['idusers'] == id_user, 'nickname'] = new_nick
        print(f"✓ Nickname untuk {id_user} berhasil diupdate menjadi: {new_nick}")

print("\n--- Selesai. Silakan cek kembali data Anda ---")

✓ Nickname untuk U00001 berhasil diupdate menjadi: ADMINISTRATOR
✓ Nickname untuk U00015 berhasil diupdate menjadi: Ika
✓ Nickname untuk U00028 berhasil diupdate menjadi: pengajar
✓ Nickname untuk U00040 berhasil diupdate menjadi: Agung
✓ Nickname untuk U00046 berhasil diupdate menjadi: sosmed
✓ Nickname untuk U00048 berhasil diupdate menjadi: Yerly
✓ Nickname untuk U00053 berhasil diupdate menjadi: Ann

--- Selesai. Silakan cek kembali data Anda ---


In [15]:
# ---------------------------------------------------------
# VERIFIKASI PERUBAHAN NICKNAME MANUAL
# ---------------------------------------------------------
# Jalankan sel ini untuk melihat hasil input manualmu tadi

target_ids = ['U00001', 'U00015', 'U00028', 'U00040', 'U00046', 'U00048', 'U00053']

# Ambil data terbaru dari df_old dan sandingkan dengan df_new
verification_df = pd.merge(
    df_old['karyawan'][df_old['karyawan']['idusers'].isin(target_ids)][['idusers', 'nickname']].copy(),
    df_new['users'][['id_user', 'name']].copy(),
    left_on='idusers',
    right_on='id_user',
    how='inner'
)

print("--- Hasil Verifikasi Nickname Manual ---")
display(verification_df[['id_user', 'name', 'nickname']])

--- Hasil Verifikasi Nickname Manual ---


,id_user,name,nickname
0,U00001,ADMINISTRATOR,ADMINISTRATOR
1,U00015,Ika Asriani Yadin,Ika
2,U00028,pengajar,pengajar
3,U00040,"Agung Wijayanto, S.T.",Agung
4,U00046,sosialmedia1,sosmed
5,U00048,"Yerly A Datu, S.Pd, M.Pd, CT.ACT, C.PS",Yerly
6,U00053,Ann Clariss Agbalog Amora,Ann


## Benerin struktur idkariawan

In [16]:
# Create a new dictionary with the updated 'idusers' values
df_new_users_dict = df_merged_users.set_index('id_user')['thnbekerja'].to_dict()

# Replace the 'idusers' values in df_old['karyawan'] with the corresponding values from df_new_users_dict
df_old['karyawan']['thnbekerja'] = df_old['karyawan']['idusers'].map(df_new_users_dict)

# Display semua baris dan kolom
print("\n5. TAMPILAN SEMUA DATA:")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_old['karyawan'])


5. TAMPILAN SEMUA DATA:


,idkaryawan,ktp,nickname,kota,tgl,jk,goldar,agama,status,alamatktp,domisili,warga,anakke,hobi,linkedin,riwayat,email,emailkantor,telp,npwp,bpjskerja,bpjssehat,idusers,nama,anak,rekening,moda,ig,fb,link,thnbekerja
0,LEAP001VI02,None,ADMINISTRATOR,None,None,Wanita,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,U00001,None,None,None,None,None,None,,2023-06-02
1,LEAP003III2023,3514186411980002,Graciela,Sidoarjo,11/24/1998,Wanita,-,Islam,Belum Kawin,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",Indonesia,1,Membaca,https://www.linkedin.com/in/graciela-evanda-ronadi-b134221a3/?originalSubdomain=id,"Maag, tipes",graciela.ronadi12@gmail.com,graciela@leapsurabaya.sch.id,0812-3447-7137,53.856.239.8-619.000,,,U00003,Graciela Evanda Ronadi,,0140881385821,Motor Pribadi,https://www.instagram.com/gracielaevr/,,https://drive.google.com/drive/folders/1alw9SubNFKSs40YOrz36JlZCGj7T8waC?usp=sharing,2023-03-13
2,LEAP011XII01,3515135105910001,DANIAR,SURABAYA,05/11/1991,Wanita,B,ISLAM,KAWIN,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,INDONESIA,1,"MAIN GAME, NONTON",https://www.linkedin.com/in/daniar-aulia-rizki-28b8a4250/,PREKLAMSIA,daniarodriscoll@gmail.com,daniar.rizki@leapsurabaya.sch.id,0896-9632-0278,46.770.512.5-603.000,21034278636,0002319622514,U00011,DANIAR AULIA RIZKI,1,1410022279715,SEPEDA MOTOR,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va7B0PtK57XOZrl-1nvu1n3Wz1jOj75r,2020-12-01
3,LEAP012III02,3578106705930001,Habibah,Semarang,05/27/1993,Wanita,AB,Islam,Belum menikah,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,Indonesia,1,Mereview berbagai jenis makanan,https://www.linkedin.com/in/habibah-melyna/,"Alergi udang, pengawet makanan dan micin",habibahmelyna@gmail.com,habibah.elfiani@leapsurabaya.sch.id,0857-3093-3317,96.917.411.9-619.000,21034278651,0002999218792,U00012,Habibah Melyna Elfiani,,1420018314442,Sepeda Motor Pribadi,https://instagram.com/habibahmelyna?igshid=MzNlNGNkZWQ4Mg==,,None,2020-03-02
4,LEAP014VII31,3578035706820005,Laksmi,Surabaya,06/17/1982,Wanita,O,ISLAM,Single,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,WNI,2,"Hiking, Reading, Traveling, Attending Concert",https://www.linkedin.com/in/laksmi-puspitowardhani-ab1a1336,Liver,laksmi.p@leapsurabaya.sch.id,laksmi.p@leapsurabaya.sch.id,6282-1399-6681,35.227.276.9-615.000,18088838356,00015384308,U00014,Laksmi Puspitowardhani,,1420016807967,Motor,https://www.instagram.com/laksmi_purplespace/?hl=en,-,https://drive.google.com/drive/folders/1-7iI4-phQIGBra_G-bekA_2Fuu_-nAiu?usp=drive_link,2018-07-31
5,LEAP015II2011,None,Ika,None,None,Wanita,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,U00015,None,None,None,None,None,None,None,14-2-2011
6,LEAP016VI2017,3524035806960001,Luluk,Lamongan,06/18/1996,Wanita,-,Islam,Belum Kawin,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",Indonesia,satu,Menonton Drama/Film,https://www.linkedin.com/in/luluk-fatikah-sari-a74001252/,Tipes dan sakit lambung,lulukfatikahsari@gmail.com,luluk.sari@leapsurabaya.sch.id,0857-8546-7685,81.269.150.9-645.000,21034278685,0002226118184,U00016,Luluk Fatikah Sari,,1420015479032,Motor Pribadi,https://www.instagram.com/lulukfatikah/,-,https://drive.google.com/drive/folders/1CF3rrcfAQzTvTxqt64FWdYP6meSOesBW?usp=sharing,2017-06-19
7,LEAP018IV20,3515165701920002,Tari,Surabaya,01/17/1992,Wanita,B,Islam,Lajang,"Tebel Timur 004/006, TEBEL, GEDANGAN, SIDOARJO","Tebel Timur JL. RA Mustika III 004/006 No. 64, TEBEL, GEDANGAN, SIDOARJO",Indonesia,1,Memasak dan membuat kerajinan tangan,ww,Maag dan darah rendah,ditari@leapsurabaya.sch.id,,0813-5966-3659,,,,U00018,Ditari Kurnia Damayanti,,1410022278949,Motor Pribadi,ww,,None,2021-04-20
8,LEAP019VI2019,3524094707820003,Tatik,Lamongan,07/07/1982,Wanita,O,Islam,Menikah,"Kebalan Kulon Rt 10 RW 02, Seka

In [17]:
import pandas as pd

# Membuat kolom baru untuk konversi bulan menjadi angka romawi
df_old['karyawan']['bulan_romawi'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).strftime('%B').upper())

# Merubah setiap bulan menjadi angka romawi
bulan_romawi_mapping = {
    'JANUARY': 'I',
    'FEBRUARY': 'II',
    'MARCH': 'III',
    'APRIL': 'IV',
    'MAY': 'V',
    'JUNE': 'VI',
    'JULY': 'VII',
    'AUGUST': 'VIII',
    'SEPTEMBER': 'IX',
    'OCTOBER': 'X',
    'NOVEMBER': 'XI',
    'DECEMBER': 'XII'
}

df_old['karyawan']['bulan_romawi'] = df_old['karyawan']['bulan_romawi'].replace(bulan_romawi_mapping)

# Membuat kolom baru untuk tanggal dan tahun
df_old['karyawan']['tanggal'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).strftime('%d'))
df_old['karyawan']['tahun'] = df_old['karyawan']['thnbekerja'].apply(lambda x: pd.Period(x).year)

# Update df_old['karyawan']['idkaryawan'] dengan struktur penamaan yang diinginkan

df_old['karyawan']['idkaryawan'] = df_old['karyawan']['idkaryawan'].str[:7] + df_old['karyawan']['tanggal'] + df_old['karyawan']['bulan_romawi'] + df_old['karyawan']['tahun'].astype(str).str[-2:]

# Display only the 'id_karyawan' and 'nama_karyawan' columns
print("\n5. TAMPILAN SEMUA DATA:")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_old['karyawan'][['idkaryawan', 'nickname', 'thnbekerja']])


5. TAMPILAN SEMUA DATA:


,idkaryawan,nickname,thnbekerja
0,LEAP00102VI23,ADMINISTRATOR,2023-06-02
1,LEAP00313III23,Graciela,2023-03-13
2,LEAP01101XII20,DANIAR,2020-12-01
3,LEAP01202III20,Habibah,2020-03-02
4,LEAP01431VII18,Laksmi,2018-07-31
5,LEAP01514II11,Ika,14-2-2011
6,LEAP01619VI17,Luluk,2017-06-19
7,LEAP01820IV21,Tari,2021-04-20
8,LEAP01901VI19,Tatik,2019-06-01
9,LEAP02030IV09,MJ,2009-04-30


## Benerin Enum dan missing value or error value

In [18]:
# Define a dictionary mapping the provided options to their corresponding values
bpjskerja_mapping = {
    '21034278636': '21034278636',
    '21034278651': '21034278651',
    '18088838356': '18088838356',
    '21034278685': '21034278685',
    '17008686762': '17008686762',
    '21034278669': '21034278669',
    '21056548478': '21056548478',
    '23182511560': '23182511560',
    '35782566048': '35782566048',
    '17008687224': '17008687224',
    '23074206378': '23074206378',
    '17008685640': '17008685640',
    '21012951303': '21012951303',
    '23153213873': '23153213873'
}

# Create a new column 'bpjskerja_mapped' with the mapped values
df_old['karyawan']['bpjskerja'] = df_old['karyawan']['bpjskerja'].map(bpjskerja_mapping)

# Verifikasi hasil
df_old['karyawan']['bpjskerja']

0             NaN
1             NaN
2     21034278636
3     21034278651
4     18088838356
5             NaN
6     21034278685
7             NaN
8             NaN
9     17008686762
10    21034278669
11    21056548478
12            NaN
13            NaN
14            NaN
15            NaN
16            NaN
17    23182511560
18    35782566048
19    17008687224
20    23074206378
21            NaN
22    17008685640
23            NaN
24            NaN
25            NaN
26    21012951303
27            NaN
28            NaN
29            NaN
30            NaN
31            NaN
32            NaN
33            NaN
34            NaN
35            NaN
36            NaN
37            NaN
38            NaN
39            NaN
40            NaN
41            NaN
42    23153213873
43            NaN
44            NaN
45            NaN
46            NaN
47            NaN
48            NaN
49            NaN
50            NaN
51            NaN
Name: bpjskerja, dtype: object

In [19]:
# Define a dictionary mapping the provided options to their corresponding values
anakke_mapping = {
    '1': 1,
    '2': 2,
    'NaN': None,
    '3': 3,
    '-': None,
    'satu': 1,
    '4': 4,
    'Pertama ': 1
}

# Create a new column 'anakke' in df_old['karyawan'] with the provided options
df_old['karyawan']['anakke'] = df_old['karyawan']['anakke'].replace(anakke_mapping)

# Verifikasi hasil
print(df_old['karyawan']['anakke'].value_counts())

anakke
1.0    28
2.0     9
3.0     4
4.0     1
Name: count, dtype: int64


In [20]:
# Define a dictionary mapping the provided options to their corresponding values
warga_mapping = {
    'Indonesia ': 'WNI',
    'Indonesia': 'WNI',
    'NaN': None,
    'WNI': 'WNI',
    'INDONESIA': 'WNI',
    '-': None,
    'fghn': None,
    'Surabaya': 'WNI'
}

# Create a new column 'warga' in df_old['karyawan'] with the provided options
df_old['karyawan']['warga'] = df_old['karyawan']['warga'].replace(warga_mapping)

# Verifikasi hasil
print(df_old['karyawan']['warga'].value_counts())

warga
WNI    43
Name: count, dtype: int64


In [21]:
# Define a dictionary mapping the provided options to their corresponding values
goldar_mapping = {
    'O': 'O',
    'B': 'B',
    '-': None,
    'AB': 'AB',
    'Ti': None,
    'O+': 'O+',
    '0': None,
    'A': 'A',
    'fg': None
}

# Create a new column 'goldar' in df_old['karyawan'] with the provided options
df_old['karyawan']['goldar'] = df_old['karyawan']['goldar'].replace(goldar_mapping)

# Verifikasi hasil
print(df_old['karyawan']['goldar'].value_counts())

goldar
O     16
B     10
AB     2
O+     2
A      2
Name: count, dtype: int64


In [22]:
# a ini mau penggolongannya seperti apa?

display(df_old['karyawan']['moda'].value_counts())
# Define a dictionary mapping old values to new values
moda_mapping = {
    'Motor Pribadi':  'Sepeda Motor',
    'Motor pribadi':  'Sepeda Motor',
    'Motor':  'Sepeda Motor',
    '-': None,
    'SEPEDA MOTOR': 'Sepeda Motor',
    'Sepeda Motor Pribadi': 'Sepeda Motor',
    'ghj': None,
    'Sepeda motor': 'Sepeda Motor',
    'Mobil': 'Mobil',
    'sepeda motor': 'Sepeda Motor',
    'Transportasi umum': 'Transportasi Umum',
    'Personal Motor':  'Sepeda Motor',
    'Motor Pribadi':  'Sepeda Motor',
    'Jalan Kaki': 'Jalan Kaki',
    'grab / mobil pribadi': 'Grab / Mobil Pribadi',
    'Sepeda kayuh': 'Sepeda Kayuh',
    'Grab Bike / Car': 'Grab',
    'Tidak ada': None
}

# Use replace() to fix the values in 'moda' column
df_old['karyawan']['moda'] = df_old['karyawan']['moda'].replace(moda_mapping)

# Verifikasi hasil
display(df_old['karyawan']['moda'].value_counts())

moda
Motor Pribadi           17
Motor pribadi            8
Motor                    4
-                        2
SEPEDA MOTOR             1
Sepeda Motor Pribadi     1
ghj                      1
Sepeda motor             1
Mobil                    1
sepeda motor             1
Transportasi umum        1
Personal Motor           1
Motor Pribadi            1
Jalan Kaki               1
grab / mobil pribadi     1
Sepeda kayuh             1
Grab Bike / Car          1
Tidak ada                1
motor pribadi            1
Name: count, dtype: int64

moda
Sepeda Motor            34
Mobil                    1
Transportasi Umum        1
Motor Pribadi            1
Jalan Kaki               1
Grab / Mobil Pribadi     1
Sepeda Kayuh             1
Grab                     1
motor pribadi            1
Name: count, dtype: int64

In [23]:
# Menggunakan replace()
df_old['karyawan']['jk'] = df_old['karyawan']['jk'].replace({
    'Wanita': 'Perempuan',
    'Pria': 'Laki laki',
    'Laki - Laki': 'Laki laki'
})

# Verifikasi hasil
print(df_old['karyawan']['jk'].value_counts())

jk
Perempuan    35
Laki laki    13
Name: count, dtype: int64


In [24]:
# Standardize agama (religion) values
agama_mapping = {
    'Islam': 'Islam',
    'ISLAM': 'Islam',
    'Kristen': 'Kristen Protestan',
    'Katholik': 'Katolik',
    'Katolik': 'Katolik',
    'Hindu': 'Hindu',
    'djhxt': None,
    'Buddha': 'Buddha',
    'Konghucu': 'Konghucu',
    '-': None,  # Convert '-' to NaN
}

df_old['karyawan']['agama'] = df_old['karyawan']['agama'].replace(agama_mapping)

# Map any remaining values to None (NaN)
df_old['karyawan']['agama'] = df_old['karyawan']['agama'].apply(
    lambda x: x if x in agama_mapping.values() else None
)

df_old['karyawan']['agama'].value_counts()

agama
Islam                35
Katolik               3
Kristen Protestan     1
Hindu                 1
Name: count, dtype: int64

In [25]:
# Standardize status pernikahan (marital status) values
status_mapping = {
    'Menikah': 'Menikah',
    'KAWIN': 'Menikah',
    'Kawin': 'Menikah',
    'Belum Menikah': 'Belum Menikah',
    'Belum menikah': 'Belum Menikah',
    'BELUM KAWIN': 'Belum Menikah',
    'Belum Kawin': 'Belum Menikah',
    'Belum kawin': 'Belum Menikah',
    'Single ': 'Belum Menikah',
    'Single': 'Belum Menikah',
    'Lajang': 'Belum Menikah',
    'Belum': 'Belum Menikah',
    'Belum nikah ': 'Belum Menikah',
    '-': None,
    'dxhj': None,
}

df_old['karyawan']['status'] = df_old['karyawan']['status'].replace(status_mapping)

# Verify results
df_old['karyawan']['status'].value_counts()

status
Belum Menikah    31
Menikah          12
Name: count, dtype: int64

In [26]:
# Standardize status values (Aktif -> 1, Non Aktif -> 0)
df_merged_users['status'] = df_merged_users['status'].replace({
    'Aktif': 1,
    'Non Aktif': 0
})

# Verify results
df_merged_users['status'].value_counts()

status
1    40
0    12
Name: count, dtype: int64

In [27]:
display(df_old['karyawan'])
display(df_merged_users)
# Display semua baris dan kolom
print("\n5. TAMPILAN SEMUA DATA:")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_old['karyawan'][['idkaryawan', 'alamatktp', 'domisili', 'kota', ]])

,idkaryawan,ktp,nickname,kota,tgl,jk,goldar,agama,status,alamatktp,domisili,warga,anakke,hobi,linkedin,riwayat,email,emailkantor,telp,npwp,bpjskerja,bpjssehat,idusers,nama,anak,rekening,moda,ig,fb,link,thnbekerja,bulan_romawi,tanggal,tahun
0,LEAP00102VI23,None,ADMINISTRATOR,None,None,Perempuan,None,None,None,None,None,None,NaN,None,None,None,None,None,None,None,NaN,None,U00001,None,None,None,None,None,None,,2023-06-02,VI,02,2023
1,LEAP00313III23,3514186411980002,Graciela,Sidoarjo,11/24/1998,Perempuan,None,Islam,Belum Menikah,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",WNI,1.0,Membaca,https://www.linkedin.com/in/graciela-evanda-ronadi-b134221a3/?originalSubdomain=id,"Maag, tipes",graciela.ronadi12@gmail.com,graciela@leapsurabaya.sch.id,0812-3447-7137,53.856.239.8-619.000,NaN,,U00003,Graciela Evanda Ronadi,,0140881385821,Sepeda Motor,https://www.instagram.com/gracielaevr/,,https://drive.google.com/drive/folders/1alw9SubNFKSs40YOrz36JlZCGj7T8waC?usp=sharing,2023-03-13,III,13,2023
2,LEAP01101XII20,3515135105910001,DANIAR,SURABAYA,05/11/1991,Perempuan,B,Islam,Menikah,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,WNI,1.0,"MAIN GAME, NONTON",https://www.linkedin.com/in/daniar-aulia-rizki-28b8a4250/,PREKLAMSIA,daniarodriscoll@gmail.com,daniar.rizki@leapsurabaya.sch.id,0896-9632-0278,46.770.512.5-603.000,21034278636,0002319622514,U00011,DANIAR AULIA RIZKI,1,1410022279715,Sepeda Motor,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va7B0PtK57XOZrl-1nvu1n3Wz1jOj75r,2020-12-01,XII,01,2020
3,LEAP01202III20,3578106705930001,Habibah,Semarang,05/27/1993,Perempuan,AB,Islam,Belum Menikah,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,WNI,1.0,Mereview berbagai jenis makanan,https://www.linkedin.com/in/habibah-melyna/,"Alergi udang, pengawet makanan dan micin",habibahmelyna@gmail.com,habibah.elfiani@leapsurabaya.sch.id,0857-3093-3317,96.917.411.9-619.000,21034278651,0002999218792,U00012,Habibah Melyna Elfiani,,1420018314442,Sepeda Motor,https://instagram.com/habibahmelyna?igshid=MzNlNGNkZWQ4Mg==,,None,2020-03-02,III,02,2020
4,LEAP01431VII18,3578035706820005,Laksmi,Surabaya,06/17/1982,Perempuan,O,Islam,Belum Menikah,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,WNI,2.0,"Hiking, Reading, Traveling, Attending Concert",https://www.linkedin.com/in/laksmi-puspitowardhani-ab1a1336,Liver,laksmi.p@leapsurabaya.sch.id,laksmi.p@leapsurabaya.sch.id,6282-1399-6681,35.227.276.9-615.000,18088838356,00015384308,U00014,Laksmi Puspitowardhani,,1420016807967,Sepeda Motor,https://www.instagram.com/laksmi_purplespace/?hl=en,-,https://drive.google.com/drive/folders/1-7iI4-phQIGBra_G-bekA_2Fuu_-nAiu?usp=drive_link,2018-07-31,VII,31,2018
5,LEAP01514II11,None,Ika,None,None,Perempuan,None,None,None,None,None,None,NaN,None,None,None,None,None,None,None,NaN,None,U00015,None,None,None,None,None,None,None,14-2-2011,II,14,2011
6,LEAP01619VI17,3524035806960001,Luluk,Lamongan,06/18/1996,Perempuan,None,Islam,Belum Menikah,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",WNI,1.0,Menonton Drama/Film,https://www.linkedin.com/in/luluk-fatikah-sari-a74001252/,Tipes dan sakit lambung,lulukfatikahsari@gmail.com,luluk.sari@leapsurabaya.sch.id,0857-8546-7685,81.269.150.9-645.000,21034278685,0002226118184,U00016,Luluk Fatikah Sari,,1420015479032,Sepeda Motor,https://www.instagram.com/lulukfatikah/,-,https://drive.google.com/drive/folders/1CF3rrcfAQzTvTxqt64FWdYP6meSOesBW?usp=sharing,2017-06-19,VI,19,2017
7,LEAP01820IV21,3515165701920002,Tari,Surabaya,01/17/1992,Perempuan,B,Islam,Belum Menikah,"Tebel Timur 004/006, TEBEL, GEDANGAN, SIDOARJO","Tebel Timur JL. RA Mustika III 004/006 No. 64, TEBEL, GEDANGAN, SIDOARJO",WNI,1.0,Memasak dan membuat kerajinan tangan,ww,Maag dan darah rendah,ditari@leapsurabaya.sch.id,,0813-5966-3659,,NaN,,U00018,Ditari Kurnia Damayanti,,1410022278949,Se

,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at,idusers,foto,idrole,wa,thnbekerja,idjabatan,idjamkerja,minat,status,idbidang,ispurchase,isteaching,ishr,isga,isit,ispdd,isbusdev,ispimpinan,ttd,expertise
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00001,logo.png,R00001,0851-7438-7539,2023-06-02,J00005,2.0,bekerja,1,NaN,0.0,0.0,0,1,0,0,0,0,None,None
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,2026-06-23 21:40:16,qWmlbcVjYmo%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00003,1707279559_ec864cc58f50e8b890d5.jpg,R00003,0812-3447-7137,2023-03-13,J00006,2.0,"Web, php, CI4, database",1,NaN,0.0,1.0,0,0,0,0,0,0,1702002184_d82747bcbd0f2bdab0a7.png,None
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00011,1692700456_2d6352eb557d734e53b7.jpeg,R00006,0896-9632-0278,2020-12-01,J00010,2.0,BERNYANYI,0,7.0,0.0,0.0,0,0,0,0,1,0,None,
3,U00012,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id,2026-06-23 21:40:16,o56YqZJkZA%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00012,1686127161_1ec3d11da554fb668e7c.jpg,R00003,0857-3093-3317,2020-03-02,J00002,3.0,Belajar dan bertumbuh,1,NaN,0.0,1.0,0,0,0,1,1,0,1702892302_89507e3f2d87fb0dfdee.png,desain grafis
4,U00014,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id,2026-06-23 21:40:16,aWlpbw%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00014,1696338938_2a535db26e3f89919325.jpg,R00008,0821-3996-6817,2018-07-31,J00004,3.0,Psikologi & Bisnis,1,NaN,0.0,0.0,0,0,0,1,0,0,None,
5,U00015,Ika Asriani Yadin,ikayadin@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00015,1694397431_30b88a94a51ff7c8024e.png,R00004,0852-1124-3425,14-2-2011,J00013,3.0,-,1,NaN,NaN,NaN,0,0,0,0,0,0,None,None
6,U00016,"Luluk Fatikah Sari, S.Pd.",luluk.sari@leapsurabaya.sch.id,2026-06-23 21:40:16,o66jrsxjY2s%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00016,1688370880_b894a9a6a8aa56770eaf.jpeg,R00006,0857-8546-7685,2017-06-19,J00007,3.0,Edu,1,NaN,0.0,1.0,0,0,0,0,0,0,1708307211_3e160b7a0d1560febc2c.png,None
7,U00018,Ditari Kurnia,admin@gmail.com,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00018,1742411840_4c3bd3fb30f609808130.shtml,R00001,0851-7438-7539,2021-04-20,J00005,3.0,bahagia dunia akhirat,1,NaN,1.0,0.0,0,1,0,0,0,0,None,None
8,U00019,"Hartatik, S.S.",hartatik@leapsurabaya.sch.id,2026-06-23 21:40:16,p5qenpJkZA%3D%3D,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00019,1690430190_e05a6e1833612e00b30a.jpeg,R00005,0852-3471-6582,2019-06-01,J00010,2.0,-,1,NaN,0.0,0.0,0,0,0,0,0,0,1708311112_ea63d8f1f2fb12dc6dbe.png,None
9,U00020,Juni Arlianto,juni.arlianto@leapsurabaya.sch.id,2026-06-23 21:40:16,aGtq,None,2026-06-23 21:40:16,2026-06-23 21:40:16,U00020,1688629580_4df06718a9d1efc1946a.jpeg,R00002,0881-0264-0001,2009-04-30,J00010,2.0,-,1,NaN,0.0,0.0,0,0,0,1,0,0,None,



5. TAMPILAN SEMUA DATA:


,idkaryawan,alamatktp,domisili,kota
0,LEAP00102VI23,None,None,None
1,LEAP00313III23,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",Sidoarjo
2,LEAP01101XII20,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,SURABAYA
3,LEAP01202III20,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,Semarang
4,LEAP01431VII18,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,Surabaya
5,LEAP01514II11,None,None,None
6,LEAP01619VI17,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",Lamongan
7,LEAP01820IV21,"Tebel Timur 004/006, TEBEL, GEDANGAN, SIDOARJO","Tebel Timur JL. RA Mustika III 004/006 No. 64, TEBEL, GEDANGAN, SIDOARJO",Surabaya
8,LEAP01901VI19,"Kebalan Kulon Rt 10 RW 02, Sekaran - Lamongan",Grand Surya blok A6 - 17 Buduran Sidoarjo,Lamongan
9,LEAP02030IV09,Amir Mahmud IV No.9-C,Amir Mahmud IV No.9-C,Surabaya


In [28]:
df_new['karyawan'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_karyawan           0 non-null      object
 1   id_user               0 non-null      object
 2   kode_karyawan         0 non-null      object
 3   nik_ktp               0 non-null      object
 4   nama_lengkap          0 non-null      object
 5   nama_panggilan        0 non-null      object
 6   tempat_lahir          0 non-null      object
 7   tanggal_lahir         0 non-null      object
 8   jenis_kelamin         0 non-null      object
 9   golongan_darah        0 non-null      object
 10  agama                 0 non-null      object
 11  status_pernikahan     0 non-null      object
 12  alamat_ktp            0 non-null      object
 13  alamat_domisili       0 non-null      object
 14  kewarganegaraan       0 non-null      object
 15  anak_ke               0 non-null      object
 16  ju

## Pindah ke df_new[karyawan]
Section ini digunakan untuk menganalisis pemetaan kolom dari `df_old['karyawan']` dan `df_merged_users` ke target `df_new['karyawan']`.

In [29]:
import pandas as pd

def analyze_migration_readiness(df_old_k, df_old_u, df_new_target):
    old_cols = set(df_old_k.columns)
    user_cols = set(df_old_u.columns)
    new_cols = set(df_new_target.columns)
    
    print("=== ANALISIS MAPPING KOLOM KARYAWAN ===\n") 
    
    # 1. Direct Match
    direct_matches = old_cols.intersection(new_cols)
    print(f"[DIRECT MATCH] Ada {len(direct_matches)} kolom yang namanya sama:")
    print(f"{list(direct_matches)}\n")
    
    # 2. Missing in Old (Need to find source or fill with default)
    missing_in_old = new_cols - old_cols
    
    # Check if they exist in user table
    from_user = missing_in_old.intersection(user_cols)
    still_missing = missing_in_old - from_user
    
    print(f"[FROM USER DATA] {len(from_user)} kolom target ada di data users:")
    print(f"{list(from_user)}\n")
    
    print(f"[STILL MISSING] {len(still_missing)} kolom target tidak ada di data karyawan lama maupun users:")
    print(f"{list(still_missing)}\n")
    
    # 3. Potential Mappings (Fuzzy matching names)
    print("[POTENTIAL MAPPINGS] Mencoba mencocokkan nama kolom (case-insensitive/underscore):")
    potential = []
    for m in still_missing:
        normalized_m = m.lower().replace('_', '').replace(' ', '')
        for o in old_cols:
            normalized_o = o.lower().replace('_', '').replace(' ', '')
            if normalized_m == normalized_o or normalized_m in normalized_o or normalized_o in normalized_m:
                potential.append(f"{o} -> {m}")
    
    if potential:
        for p in set(potential):
            print(f"  - {p}")
    else:
        print("  - Tidak ditemukan kecocokan otomatis.")

analyze_migration_readiness(df_old['karyawan'], df_merged_users, df_new['karyawan'])


=== ANALISIS MAPPING KOLOM KARYAWAN ===

[DIRECT MATCH] Ada 2 kolom yang namanya sama:
['hobi', 'agama']

[FROM USER DATA] 1 kolom target ada di data users:
['id_user']

[STILL MISSING] 34 kolom target tidak ada di data karyawan lama maupun users:
['status_aktif', 'nomor_telepon', 'nomor_npwp', 'nomor_rekening', 'id_shift', 'akun_linkedin', 'foto_profile', 'moda_transportasi', 'akun_instagram', 'status_pernikahan', 'id_karyawan', 'ttd_digital', 'tanggal_lahir', 'anak_ke', 'nik_ktp', 'golongan_darah', 'link_dokumen_pribadi', 'akun_facebook', 'nama_panggilan', 'alamat_ktp', 'nama_lengkap', 'tempat_lahir', 'keahlian', 'jenis_kelamin', 'tanggal_bergabung', 'email_pribadi', 'riwayat_kesehatan', 'kode_karyawan', 'alamat_domisili', 'kewarganegaraan', 'email_kantor', 'bpjs_ketenagakerjaan', 'bpjs_kesehatan', 'jumlah_anak']

[POTENTIAL MAPPINGS] Mencoba mencocokkan nama kolom (case-insensitive/underscore):
  - anakke -> anak_ke
  - ktp -> alamat_ktp
  - npwp -> nomor_npwp
  - nama -> nama_pangg

In [30]:
# 1. Ambil kolom yang diperlukan dari df_merged_users sebagai referensi update
# Pastikan menggunakan kolom 'nama' (atau 'name' jika 'nama' tidak ada) dan 'email'
cols_to_use = ['idusers', 'email']
if 'nama' in df_merged_users.columns:
    cols_to_use.append('nama')
elif 'name' in df_merged_users.columns:
    # Jika di df_merged_users namanya 'name', kita rename jadi 'nama' agar sesuai target
    df_update_ref = df_merged_users[['idusers', 'name', 'email']].rename(columns={'name': 'nama'})
    cols_to_use = None # Sudah dihandle di baris atas

if cols_to_use:
    df_update_ref = df_merged_users[cols_to_use]

# 2. Hapus kolom 'nama' dan 'email' lama di df_old['karyawan'] 
# agar tidak terjadi duplikasi kolom saat merge
df_old['karyawan'] = df_old['karyawan'].drop(columns=['nama', 'email'])

# 3. Merge df_old['karyawan'] dengan data referensi berdasarkan 'idusers'
df_old['karyawan'] = df_old['karyawan'].merge(
    df_update_ref, 
    on='idusers', 
    how='left'
)

# Opsional: Cek hasil merge
print("Update selesai. Preview df_old['karyawan']:")
display(df_old['karyawan'])


Update selesai. Preview df_old['karyawan']:


,idkaryawan,ktp,nickname,kota,tgl,jk,goldar,agama,status,alamatktp,domisili,warga,anakke,hobi,linkedin,riwayat,emailkantor,telp,npwp,bpjskerja,bpjssehat,idusers,anak,rekening,moda,ig,fb,link,thnbekerja,bulan_romawi,tanggal,tahun,nama,email
0,LEAP00102VI23,None,ADMINISTRATOR,None,None,Perempuan,None,None,None,None,None,None,NaN,None,None,None,None,None,None,NaN,None,U00001,None,None,None,None,None,,2023-06-02,VI,02,2023,ADMINISTRATOR,ditari@leapsurabaya.sch.id
1,LEAP00313III23,3514186411980002,Graciela,Sidoarjo,11/24/1998,Perempuan,None,Islam,Belum Menikah,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",WNI,1.0,Membaca,https://www.linkedin.com/in/graciela-evanda-ronadi-b134221a3/?originalSubdomain=id,"Maag, tipes",graciela@leapsurabaya.sch.id,0812-3447-7137,53.856.239.8-619.000,NaN,,U00003,,0140881385821,Sepeda Motor,https://www.instagram.com/gracielaevr/,,https://drive.google.com/drive/folders/1alw9SubNFKSs40YOrz36JlZCGj7T8waC?usp=sharing,2023-03-13,III,13,2023,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id
2,LEAP01101XII20,3515135105910001,DANIAR,SURABAYA,05/11/1991,Perempuan,B,Islam,Menikah,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,WNI,1.0,"MAIN GAME, NONTON",https://www.linkedin.com/in/daniar-aulia-rizki-28b8a4250/,PREKLAMSIA,daniar.rizki@leapsurabaya.sch.id,0896-9632-0278,46.770.512.5-603.000,21034278636,0002319622514,U00011,1,1410022279715,Sepeda Motor,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va7B0PtK57XOZrl-1nvu1n3Wz1jOj75r,2020-12-01,XII,01,2020,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id
3,LEAP01202III20,3578106705930001,Habibah,Semarang,05/27/1993,Perempuan,AB,Islam,Belum Menikah,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,WNI,1.0,Mereview berbagai jenis makanan,https://www.linkedin.com/in/habibah-melyna/,"Alergi udang, pengawet makanan dan micin",habibah.elfiani@leapsurabaya.sch.id,0857-3093-3317,96.917.411.9-619.000,21034278651,0002999218792,U00012,,1420018314442,Sepeda Motor,https://instagram.com/habibahmelyna?igshid=MzNlNGNkZWQ4Mg==,,None,2020-03-02,III,02,2020,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id
4,LEAP01431VII18,3578035706820005,Laksmi,Surabaya,06/17/1982,Perempuan,O,Islam,Belum Menikah,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,WNI,2.0,"Hiking, Reading, Traveling, Attending Concert",https://www.linkedin.com/in/laksmi-puspitowardhani-ab1a1336,Liver,laksmi.p@leapsurabaya.sch.id,6282-1399-6681,35.227.276.9-615.000,18088838356,00015384308,U00014,,1420016807967,Sepeda Motor,https://www.instagram.com/laksmi_purplespace/?hl=en,-,https://drive.google.com/drive/folders/1-7iI4-phQIGBra_G-bekA_2Fuu_-nAiu?usp=drive_link,2018-07-31,VII,31,2018,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id
5,LEAP01514II11,None,Ika,None,None,Perempuan,None,None,None,None,None,None,NaN,None,None,None,None,None,None,NaN,None,U00015,None,None,None,None,None,None,14-2-2011,II,14,2011,Ika Asriani Yadin,ikayadin@leapsurabaya.sch.id
6,LEAP01619VI17,3524035806960001,Luluk,Lamongan,06/18/1996,Perempuan,None,Islam,Belum Menikah,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",WNI,1.0,Menonton Drama/Film,https://www.linkedin.com/in/luluk-fatikah-sari-a74001252/,Tipes dan sakit lambung,luluk.sari@leapsurabaya.sch.id,0857-8546-7685,81.269.150.9-645.000,21034278685,0002226118184,U00016,,1420015479032,Sepeda Motor,https://www.instagram.com/lulukfatikah/,-,https://drive.google.com/drive/folders/1CF3rrcfAQzTvTxqt64FWdYP6meSOesBW?usp=sharing,2017-06-19,VI,19,2017,"Luluk Fatikah Sari, S.Pd.",luluk.sari@leapsurabaya.sch.id
7,LEAP01820IV21,3515165701920002,Tari,Surabaya,01/17/1992,Perempuan,B,Islam,Belum Menikah,"Tebel Timur 004/006, TEBEL, GEDANGAN, SIDOARJO","Tebel Timur JL. RA Mustika III 004/006 No. 64, TEBEL, GEDANGAN, SIDOARJO",WNI,1.0,Memasak dan membuat kerajinan tangan,ww,Maag dan darah r

In [31]:
import pandas as pd
import numpy as np
import datetime

# --- ANALISA DAN MIGRASI df_old['karyawan'] KE df_new['karyawan'] (FIXED DB_FUTURE) ---

# 1. Persiapkan data tambahan dari df_merged_users (seperti thnbekerja, expertise, dll)
user_cols_to_add = [
    'idusers', 'expertise', 'idjamkerja', 
    'status', 'foto', 'ttd'
]
df_user_info = df_merged_users[user_cols_to_add].rename(columns={'status': 'status_user_active'})

# Gabungkan data user ke df_old['karyawan'] agar semua data source terkumpul di satu tempat
if 'status_user_active' not in df_old['karyawan'].columns:
    df_old['karyawan'] = df_old['karyawan'].merge(df_user_info, on='idusers', how='left')

# 🆕 PEMBARUAN DATA INDUK (PRE-CLEANING SENSOR)
# Memastikan nama_lengkap tidak kosong (Guard NOT NULL)
if 'nama' in df_old['karyawan'].columns:
    df_old['karyawan']['nama'] = df_old['karyawan']['nama'].fillna('Tanpa Nama Resmi')

# Menyamakan ENUM Jenis Kelamin (Laki-laki -> Laki laki)
if 'jk' in df_old['karyawan'].columns:
    df_old['karyawan']['jk'] = df_old['karyawan']['jk'].astype(str).str.replace('-', ' ')

# 2. DEFINISI PEMETAAN KOLOM (SUDAH DISESUAIKAN KE DB_FUTURE)
column_mapping = {
    'id_karyawan': 'idkaryawan',
    'id_user': 'idusers',
    'kode_karyawan': 'idkaryawan',        # 🎯 BARU: Pindahkan idkaryawan lama ke kode_karyawan
    'nik_ktp': 'ktp',
    'nama_lengkap': 'nama',               # 🎯 BARU: Menambahkan kolom nama_lengkap yang wajib diisi
    'nama_panggilan': 'nickname',
    'tempat_lahir': 'kota',
    'tanggal_lahir': 'tgl',
    'jenis_kelamin': 'jk',
    'golongan_darah': 'goldar',
    'agama': 'agama',
    'status_pernikahan': 'status', 
    'alamat_ktp': 'alamatktp',
    'alamat_domisili': 'domisili',
    'kewarganegaraan': 'warga',
    'anak_ke': 'anakke',
    'jumlah_anak': 'anak',
    'hobi': 'hobi',
    'akun_linkedin': 'linkedin',
    'email_pribadi': 'email',
    'email_kantor': 'emailkantor',
    'nomor_telepon': 'telp',
    'nomor_npwp': 'npwp',
    'bpjs_ketenagakerjaan': 'bpjskerja',
    'bpjs_kesehatan': 'bpjssehat',
    'nomor_rekening': 'rekening',
    'moda_transportasi': 'moda',
    'akun_instagram': 'ig',
    'akun_facebook': 'fb',
    'link_dokumen_pribadi': 'link',
    'riwayat_kesehatan': 'riwayat',
    'tanggal_bergabung': 'thnbekerja',    # 🎯 BARU: thnbekerja lama dipetakan ke tanggal_bergabung (Bukan tahun_mulai_kerja)
    'keahlian': 'expertise',
    'id_shift': 'idjamkerja',
    'status_aktif': 'status_user_active', 
    'foto_profile': 'foto',
    'ttd_digital': 'ttd'
}

# 3. PROSES PEMBENTUKAN DATAFRAME BARU
df_migration = pd.DataFrame()

for target_col, source_col in column_mapping.items():
    if source_col in df_old['karyawan'].columns:
        df_migration[target_col] = df_old['karyawan'][source_col]
    else:
        print(f"Peringatan: Kolom '{source_col}' tidak ditemukan di source. Diisi NULL.")
        df_migration[target_col] = None

# 4. SIMPAN KE df_new['karyawan']
df_new['karyawan'] = df_migration

# 5. CLEANING DATA (Kosong atau '-' menjadi None)
df_new['karyawan'] = df_new['karyawan'].replace(['', '-'], None)

# 6. FORMATTING TANGGAL LAHIR & TANGGAL BERGABUNG (Menyesuaikan tipe Date MySQL)
df_new['karyawan']['tanggal_lahir'] = pd.to_datetime(df_new['karyawan']['tanggal_lahir'], errors='coerce').dt.date

# 🎯 BARU: thnbekerja dikonversi ke tipe Date penuh (YYYY-MM-DD), default ke awal tahun jika kosong
df_new['karyawan']['tanggal_bergabung'] = pd.to_numeric(df_new['karyawan']['tanggal_bergabung'], errors='coerce')
df_new['karyawan']['tanggal_bergabung'] = df_new['karyawan']['tanggal_bergabung'].apply(
    lambda x: f"{int(x)}-01-01" if pd.notna(x) else '2026-01-01'
)
df_new['karyawan']['tanggal_bergabung'] = pd.to_datetime(df_new['karyawan']['tanggal_bergabung']).dt.date

# ... (Poin 1 sampai 6 di dalam kode lambat/lama kamu tetap biarkan sama) ...

# 7. 🎯 PERBAIKAN CIMUT: CONVERT KE INTEGER MURNI & JINAKKAN FLOAT .0
# Kita paksa kolom id_shift dan anak_ke menjadi Int64 Nullable murni
df_new['karyawan']['id_shift'] = pd.to_numeric(df_new['karyawan']['id_shift'], errors='coerce').astype('Int64')
df_new['karyawan']['anak_ke'] = pd.to_numeric(df_new['karyawan']['anak_ke'], errors='coerce').astype('Int64')
df_new['karyawan']['jumlah_anak'] = pd.to_numeric(df_new['karyawan']['jumlah_anak'], errors='coerce').astype('Int64')

# 🆕 SUNTIKAN LOOPING AUTO-INCREMENT UNTUK id_karyawan 🆕
# Kita buat penomoran urut murni bawaan Python (1 sampai jumlah baris data)
# Taktik ini otomatis merontokkan float ghaib bertipe desimal .0
total_baris_karyawan = len(df_new['karyawan'])
df_new['karyawan']['id_karyawan'] = range(1, total_baris_karyawan + 1)

# Pastikan dikunci ke tipe Int64 murni agar aman terbaca oleh mysql-connector
df_new['karyawan']['id_karyawan'] = df_new['karyawan']['id_karyawan'].astype('Int64')

# 🆕 SENSOR TAMBAHAN GUNA MENJAGA ATURAN NOT NULL FIELD
df_new['karyawan']['status_pernikahan'] = df_new['karyawan']['status_pernikahan'].apply(
    lambda x: 'Belum Menikah' if pd.isna(x) or str(x).strip() in ["", "None", "nan"] else str(x).strip()
)

# 8. FINAL CLEANUP: Ganti NaN/NaT/pd.NA ke None
# Kita konversi ke .astype(object) dulu agar bisa menampung 'None' standar Python alih-alih pd.NA.
df_new['karyawan'] = df_new['karyawan'].astype(object).where(pd.notnull(df_new['karyawan']), None)

print(f"✓ Berhasil membuat looping auto-increment untuk {len(df_new['karyawan'])} data karyawan!")
print("--- Preview data hasil adaptasi id_karyawan murni (INT TANPA .0) ---")
df_new['karyawan'].info()
display(df_new['karyawan'][['id_karyawan', 'id_user', 'kode_karyawan', 'nama_lengkap']].head(5))

✓ Berhasil membuat looping auto-increment untuk 52 data karyawan!
--- Preview data hasil adaptasi id_karyawan murni (INT TANPA .0) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_karyawan           52 non-null     object
 1   id_user               52 non-null     object
 2   kode_karyawan         52 non-null     object
 3   nik_ktp               46 non-null     object
 4   nama_lengkap          52 non-null     object
 5   nama_panggilan        52 non-null     object
 6   tempat_lahir          45 non-null     object
 7   tanggal_lahir         46 non-null     object
 8   jenis_kelamin         52 non-null     object
 9   golongan_darah        32 non-null     object
 10  agama                 40 non-null     object
 11  status_pernikahan     52 non-null     object
 12  alamat_ktp            44 non-null     object
 13  alamat_

,id_karyawan,id_user,kode_karyawan,nama_lengkap
0,1,U00001,LEAP00102VI23,ADMINISTRATOR
1,2,U00003,LEAP00313III23,"Graciela Evanda Ronadi, S.Kom."
2,3,U00011,LEAP01101XII20,DANIAR AULIA RIZKI
3,4,U00012,LEAP01202III20,Habibah Melyna
4,5,U00014,LEAP01431VII18,Laksmi Puspitowardhani


In [32]:
print("\n--- Migrasi Selesai ---")
print(f"Jumlah baris yang dipindahkan: {len(df_new['karyawan'])}")
display(df_new['karyawan'])


--- Migrasi Selesai ---
Jumlah baris yang dipindahkan: 52


,id_karyawan,id_user,kode_karyawan,nik_ktp,nama_lengkap,nama_panggilan,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,agama,status_pernikahan,alamat_ktp,alamat_domisili,kewarganegaraan,anak_ke,jumlah_anak,hobi,akun_linkedin,email_pribadi,email_kantor,nomor_telepon,nomor_npwp,bpjs_ketenagakerjaan,bpjs_kesehatan,nomor_rekening,moda_transportasi,akun_instagram,akun_facebook,link_dokumen_pribadi,riwayat_kesehatan,tanggal_bergabung,keahlian,id_shift,status_aktif,foto_profile,ttd_digital
0,1,U00001,LEAP00102VI23,None,ADMINISTRATOR,ADMINISTRATOR,None,None,Perempuan,None,None,Belum Menikah,None,None,None,None,None,None,None,ditari@leapsurabaya.sch.id,None,None,None,None,None,None,None,None,None,None,None,2026-01-01,None,2,1,logo.png,None
1,2,U00003,LEAP00313III23,3514186411980002,"Graciela Evanda Ronadi, S.Kom.",Graciela,Sidoarjo,1998-11-24,Perempuan,None,Islam,Belum Menikah,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",WNI,1,None,Membaca,https://www.linkedin.com/in/graciela-evanda-ronadi-b134221a3/?originalSubdomain=id,graciela@leapsurabaya.sch.id,graciela@leapsurabaya.sch.id,0812-3447-7137,53.856.239.8-619.000,None,None,0140881385821,Sepeda Motor,https://www.instagram.com/gracielaevr/,None,https://drive.google.com/drive/folders/1alw9SubNFKSs40YOrz36JlZCGj7T8waC?usp=sharing,"Maag, tipes",2026-01-01,None,2,1,1707279559_ec864cc58f50e8b890d5.jpg,1702002184_d82747bcbd0f2bdab0a7.png
2,3,U00011,LEAP01101XII20,3515135105910001,DANIAR AULIA RIZKI,DANIAR,SURABAYA,1991-05-11,Perempuan,B,Islam,Menikah,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,WNI,1,1,"MAIN GAME, NONTON",https://www.linkedin.com/in/daniar-aulia-rizki-28b8a4250/,daniar.rizki@leapsurabaya.sch.id,daniar.rizki@leapsurabaya.sch.id,0896-9632-0278,46.770.512.5-603.000,21034278636,0002319622514,1410022279715,Sepeda Motor,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va7B0PtK57XOZrl-1nvu1n3Wz1jOj75r,PREKLAMSIA,2026-01-01,None,2,0,1692700456_2d6352eb557d734e53b7.jpeg,None
3,4,U00012,LEAP01202III20,3578106705930001,Habibah Melyna,Habibah,Semarang,1993-05-27,Perempuan,AB,Islam,Belum Menikah,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,WNI,1,None,Mereview berbagai jenis makanan,https://www.linkedin.com/in/habibah-melyna/,habibah.elfiani@leapsurabaya.sch.id,habibah.elfiani@leapsurabaya.sch.id,0857-3093-3317,96.917.411.9-619.000,21034278651,0002999218792,1420018314442,Sepeda Motor,https://instagram.com/habibahmelyna?igshid=MzNlNGNkZWQ4Mg==,None,None,"Alergi udang, pengawet makanan dan micin",2026-01-01,desain grafis,3,1,1686127161_1ec3d11da554fb668e7c.jpg,1702892302_89507e3f2d87fb0dfdee.png
4,5,U00014,LEAP01431VII18,3578035706820005,Laksmi Puspitowardhani,Laksmi,Surabaya,1982-06-17,Perempuan,O,Islam,Belum Menikah,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,WNI,2,None,"Hiking, Reading, Traveling, Attending Concert",https://www.linkedin.com/in/laksmi-puspitowardhani-ab1a1336,laksmi.p@leapsurabaya.sch.id,laksmi.p@leapsurabaya.sch.id,6282-1399-6681,35.227.276.9-615.000,18088838356,00015384308,1420016807967,Sepeda Motor,https://www.instagram.com/laksmi_purplespace/?hl=en,None,https://drive.google.com/drive/folders/1-7iI4-phQIGBra_G-bekA_2Fuu_-nAiu?usp=drive_link,Liver,2026-01-01,None,3,1,1696338938_2a535db26e3f89919325.jpg,None
5,6,U00015,LEAP01514II11,None,Ika Asriani Yadin,Ika,None,None,Perempuan,None,None,Belum Menikah,None,None,None,None,None,None,None,ikayadin@leapsurabaya.sch.id,None,None,None,None,None,None,None,None,None,None,None,2026-01-01,None,3,1,1694397431_30b88a94a51ff7c8024e.png,None
6,7,U00016,LEAP01619VI17,3524035806960001,"Luluk Fatikah Sari, S.Pd.",Luluk,Lamongan,1996-06-18,Perempuan,None,Islam,Belum Menikah,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",WNI,1,None,Menonton Drama/Film,https://www.linkedin.com/in/luluk-fatikah-sari-a74001252/,luluk.sa

In [33]:
import pandas as pd
import pickle

print("================================================================================")
print(" 📦 MEMBUAT & MENGEKSPOR KAMUS SAKTI RELASI KARYAWAN (MAPPING TABLE) 📦 ")
print("================================================================================")

df_karyawan_mapping_table = pd.DataFrame()

# 1. ID Karyawan Baru (Integer Auto-Increment)
df_karyawan_mapping_table['id_karyawan'] = df_new['karyawan']['id_karyawan'].astype('Int64')

# 2. Kode Karyawan Baru (Hasil dari pemindahan NIK/ID lama)
df_karyawan_mapping_table['kode_karyawan'] = df_new['karyawan']['kode_karyawan'].astype(str).str.strip()

# 3. ID Karyawan Lama (Varchar Mentah Langsung dari df_old) -> INI YANG DIPERBAIKI (TIDAK PAKAI UNDERSCORE)
df_karyawan_mapping_table['id_karyawan_lama'] = id_karyawan_lama.astype(str).str.strip()

# 4. ID User (Jika diperlukan untuk cross-check)
df_karyawan_mapping_table['id_user'] = df_new['karyawan']['id_user'].astype(str).str.strip()

# Tampilkan preview tabel pemetaan 
print("--- Preview Kamus Sakti Pemetaan Karyawan ---")
df_karyawan_mapping_table.info()
display(df_karyawan_mapping_table.head())

# Ekspor tabel pemetaan ini menjadi file pickle (.pkl)
mapping_file_name = 'mapping_id_karyawan.pkl'
with open(mapping_file_name, 'wb') as f:
    pickle.dump(df_karyawan_mapping_table, f)

print(f"\n💾 🎉 BERHASIL! Kamus sakti berhasil diekspor menjadi '{mapping_file_name}'!")

 📦 MEMBUAT & MENGEKSPOR KAMUS SAKTI RELASI KARYAWAN (MAPPING TABLE) 📦 
--- Preview Kamus Sakti Pemetaan Karyawan ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_karyawan       52 non-null     Int64 
 1   kode_karyawan     52 non-null     object
 2   id_karyawan_lama  52 non-null     object
 3   id_user           52 non-null     object
dtypes: Int64(1), object(3)
memory usage: 1.8+ KB


,id_karyawan,kode_karyawan,id_karyawan_lama,id_user
0,1,LEAP00102VI23,LEAP001VI02,U00001
1,2,LEAP00313III23,LEAP003III2023,U00003
2,3,LEAP01101XII20,LEAP011XII01,U00011
3,4,LEAP01202III20,LEAP012III02,U00012
4,5,LEAP01431VII18,LEAP014VII31,U00014



💾 🎉 BERHASIL! Kamus sakti berhasil diekspor menjadi 'mapping_id_karyawan.pkl'!


# Fixing Tabel keluarga_karyawan

In [34]:
import pandas as pd
import numpy as np

# ================================================================================
# 🚀 FASE 2 - STEP 2: REKAYASA RELASI & NORMALISASI KELUARGA KARYAWAN 🚀
# ================================================================================

# 1. Persiapkan mapping id_karyawan asli (ID Auto-Increment Baru) dari df_new['karyawan']
# Kita ambil jembatan id_user dan id_karyawan barunya
df_karyawan_map = df_new['karyawan'][['id_user', 'id_karyawan']].copy()

# 2. Gabungkan id_karyawan baru ke dalam df_old['keluarga'] dengan mengawinkan id_user lewat idusers lama
df_keluarga_source = df_old['keluarga'].merge(
    df_karyawan_map, 
    left_on='idusers', 
    right_on='id_user', 
    how='left'
)

# 3. DEFINISI PEMETAAN KOLOM (STRUKTUR LAMA DIADAPTASIKAN KE DB_FUTURE)
keluarga_mapping = {
    'id_keluarga': 'idkeluarga_ghaib',    # 🎯 BARU: Diarahkan ke kolom ghaib karena di MySQL baru sudah AUTO_INCREMENT
    'id_karyawan': 'id_karyawan',         # 🎯 BARU: Mengunci ID Karyawan Baru hasil perkawinan merge di atas
    'hubungan_keluarga': 'hubungan',
    'nama_lengkap': 'namalengkap',
    'pekerjaan': 'pekerjaan',
    'nomor_hp': 'hp'
}

# 3.1. NORMALISASI HUBUNGAN KELUARGA (100% AKURAT MENGGUNAKAN KAMUS ORIGINAL CIMUT)
hubungan_map = {
    'Ibu Kandung': 'Ibu', 'Ibu': 'Ibu', 
    'Ayah Kandung': 'Ayah', 'Ayah': 'Ayah', 'Bapak': 'Ayah',
    'Suami Tercintah': 'Suami', 'Suami': 'Suami', 'suami': 'Suami', 'SUAMI': 'Suami',
    'Istri': 'Istri', 'istri': 'Istri', 'ISTRI': 'Istri',
    'Adik Perempuan': 'Adik', 'Adik Laki-laki': 'Adik', 'Adik kandung': 'Adik', 'Adik': 'Adik', 'Adek Kandung': 'Adik', 'adek': 'Adik', 'adik': 'Adik',
    'Kakak Kandung': 'Kakak', 'Kaka Kandung Pertama': 'Kakak', 'Kaka Kandung Kedua': 'Kakak', 'Kakak': 'Kakak', 'kakak': 'Kakak',
    'anak': 'Anak', 'Anak': 'Anak',
    'Saudra Kandung': 'Saudara', 'Saudara Kandung': 'Saudara', 'Saudara': 'Saudara', 'Orang Tua': 'Ibu'
}

# 4. PROSES MIGRASI MENGGUNAKAN LOOPING POLA ORIGINAL CIMUT
df_keluarga_new = pd.DataFrame()
for target_col, source_col in keluarga_mapping.items():
    if source_col in df_keluarga_source.columns:
        if target_col == 'hubungan_keluarga':
            df_keluarga_new[target_col] = df_keluarga_source[source_col].map(hubungan_map).fillna('Saudara')
        else:
            df_keluarga_new[target_col] = df_keluarga_source[source_col]
    else:
        # 🎯 BARU: Jika targetnya id_keluarga, otomatis diset None agar diisi AUTO_INCREMENT oleh MySQL
        df_keluarga_new[target_col] = None

# 5. SIMPAN KE df_new['keluarga_karyawan'] DAN MEMBERSIHKAN STRING KOSONG
df_new['keluarga_karyawan'] = df_keluarga_new.replace(['', '-'], None)

# 🎯 BARU: Paksa tipe data id_karyawan menjadi Int64 bulat murni (Menjinakkan Float .0)
df_new['keluarga_karyawan']['id_karyawan'] = pd.to_numeric(df_new['keluarga_karyawan']['id_karyawan'], errors='coerce').astype('Int64')

# 🎯 BARU: Final Cleanup agar objek NaN/NaT diratakan menjadi None yang ramah dengan mysql-connector
df_new['keluarga_karyawan'] = df_new['keluarga_karyawan'].astype(object).where(pd.notnull(df_new['keluarga_karyawan']), None)

# 6. PRINT METADATA OUTPUT SEPERTI LOG LAMA KAMU
print(f"✓ Berhasil memindahkan {len(df_new['keluarga_karyawan'])} data keluarga")
display(df_new['keluarga_karyawan'])
print("\n--- Value Counts Hubungan Keluarga ---")
display(df_new['keluarga_karyawan']['hubungan_keluarga'].value_counts())
df_new['keluarga_karyawan'].info()

✓ Berhasil memindahkan 67 data keluarga


,id_keluarga,id_karyawan,hubungan_keluarga,nama_lengkap,pekerjaan,nomor_hp
0,None,2,Ibu,Nur Fatmawati,Wiraswasta,081234477137
1,None,9,Suami,Fathul Kirom,Suami,0852-3101-7799
2,None,12,Ayah,"Suwandi, S.Pd.",Guru Matematika SMAN 17 Surabaya,087853591616
3,None,12,Ibu,"Ir. Hj. Erhasyati Islamiyah, M.M.",(Pensiun) Guru Biologi SMA Muhammadiyah 2 Surabaya,08179365966
4,None,3,Suami,JONATHAN O'DRISCOLL,TIDAK BEKERJA,089696320278
5,None,12,Adik,Fahada Akbariyah,Pekerja Laboratoriumm RS Soewandhi,088801762951
6,None,12,Adik,Rafi Arif Billah,Mahasiswa S1 Manajemen Undika,08563002290
7,None,18,Ayah,Herrie Susanto,Wiraswasta,08123429138
8,None,18,Ibu,Juliani,tidak bekerja,081911541719
9,None,17,Saudara,AGUS DJOKO PRANOWO,KARYAWAN SWASTA,087852999236



--- Value Counts Hubungan Keluarga ---


hubungan_keluarga
Ibu        15
Ayah       13
Adik       10
Saudara    10
Suami       9
Anak        4
Kakak       4
Istri       2
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_keluarga        0 non-null      object
 1   id_karyawan        67 non-null     object
 2   hubungan_keluarga  67 non-null     object
 3   nama_lengkap       67 non-null     object
 4   pekerjaan          64 non-null     object
 5   nomor_hp           61 non-null     object
dtypes: object(6)
memory usage: 3.3+ KB


In [35]:
df_new['keluarga_karyawan']['hubungan_keluarga'].value_counts()

hubungan_keluarga
Ibu        15
Ayah       13
Adik       10
Saudara    10
Suami       9
Anak        4
Kakak       4
Istri       2
Name: count, dtype: int64

# Fixing table bidang_kategori, bidang_link.

In [36]:
# ---------------------------------------------------------
# MIGRASI TABEL bidang_kategori
# ---------------------------------------------------------

# 1. Definisi Mapping
bidang_kategori_mapping = {
    'id_bidang_kategori': 'idkatbid',
    'nama_kategori_bidang': 'namakatbid',
    'id_bidang': 'idbidang'
}

# 2. Proses Migrasi
df_bidang_kategori_new = pd.DataFrame()
for target_col, source_col in bidang_kategori_mapping.items():
    if source_col in df_old['bidangkategori'].columns:
        df_bidang_kategori_new[target_col] = df_old['bidangkategori'][source_col]
    else:
        df_bidang_kategori_new[target_col] = None

# 3. Simpan ke df_new
df_new['bidang_kategori'] = df_bidang_kategori_new

print(f"✓ Berhasil memindahkan {len(df_new['bidang_kategori'])} data bidang_kategori")
display(df_new['bidang_kategori'].head())
df_new['bidang_kategori'].info()

✓ Berhasil memindahkan 12 data bidang_kategori


,id_bidang_kategori,nama_kategori_bidang,id_bidang
0,7,Brand Identity,7
1,13,Training,9
2,14,Referensi,9
3,16,Training,8
4,17,Training,11


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_bidang_kategori    12 non-null     int64 
 1   nama_kategori_bidang  12 non-null     object
 2   id_bidang             12 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 420.0+ bytes


In [37]:
# ---------------------------------------------------------
# MIGRASI TABEL bidang_link
# ---------------------------------------------------------

# 1. Definisi Mapping
bidang_link_mapping = {
    'id_bidang_link': 'idformbid',
    'nama_form': 'namaformbid',
    'link_drive': 'link',
    'id_bidang_kategori': 'idkatbid',
    'status_share': 'share'
}

# 2. Proses Migrasi
df_bidang_link_new = pd.DataFrame()
for target_col, source_col in bidang_link_mapping.items():
    if source_col in df_old['bidanglink'].columns:
        if target_col == 'status_share':
            # Convert '0'/'1' or other values to int
            df_bidang_link_new[target_col] = pd.to_numeric(df_old['bidanglink'][source_col], errors='coerce').fillna(0).astype(int)
        else:
            df_bidang_link_new[target_col] = df_old['bidanglink'][source_col]
    else:
        df_bidang_link_new[target_col] = None

# 3. Simpan ke df_new
df_new['bidang_link'] = df_bidang_link_new

print(f"✓ Berhasil memindahkan {len(df_new['bidang_link'])} data bidang_link")
display(df_new['bidang_link'].head())
df_new['bidang_link'].info()

✓ Berhasil memindahkan 7 data bidang_link


,id_bidang_link,nama_form,link_drive,id_bidang_kategori,status_share
0,16,Daftar Training,https://drive.google.com/drive/folders/1BCPhp6eG5mNeWyTZS8i4xHEor-zgeyJu?usp=drive_link,13,0
1,17,Referensi,https://drive.google.com/drive/folders/1Lo6hzugBnuacIyHPF6nef-9BV1O9mnCQ?usp=drive_link,14,0
2,18,Dokumentasi,https://drive.google.com/drive/folders/14ieegP1eR_Vdj-QCOhevVeKR2DOaQ5_w?usp=drive_link,24,0
3,19,List Training,https://drive.google.com/drive/folders/1yivPYk3nJ3XmRZ5Np3S7d30AK3yw14GJ?usp=drive_link,16,0
4,20,Referensi,https://drive.google.com/drive/folders/1ah2G3XtafGmA3B1V01Np8rmv8lZrUmKl?usp=drive_link,19,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id_bidang_link      7 non-null      int64 
 1   nama_form           7 non-null      object
 2   link_drive          7 non-null      object
 3   id_bidang_kategori  7 non-null      int64 
 4   status_share        7 non-null      int64 
dtypes: int64(3), object(2)
memory usage: 412.0+ bytes


In [38]:
# Review the processed data for karyawan table
print("Processed Data for karyawan table:")
display(df_new['karyawan'])

# Review the processed data for keluarga_karyawan table
print("Processed Data for keluarga_karyawan table:")
display(df_new['keluarga_karyawan'])

# Review the processed data for bidang_kategori table
print("Processed Data for bidang_kategori table:")
display(df_new['bidang_kategori'])
display(df_old['bidang'])

# Review the processed data for bidang_link table
print("Processed Data for bidang_link table:")
display(df_new['bidang_link'])

Processed Data for karyawan table:


,id_karyawan,id_user,kode_karyawan,nik_ktp,nama_lengkap,nama_panggilan,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,agama,status_pernikahan,alamat_ktp,alamat_domisili,kewarganegaraan,anak_ke,jumlah_anak,hobi,akun_linkedin,email_pribadi,email_kantor,nomor_telepon,nomor_npwp,bpjs_ketenagakerjaan,bpjs_kesehatan,nomor_rekening,moda_transportasi,akun_instagram,akun_facebook,link_dokumen_pribadi,riwayat_kesehatan,tanggal_bergabung,keahlian,id_shift,status_aktif,foto_profile,ttd_digital
0,1,U00001,LEAP00102VI23,None,ADMINISTRATOR,ADMINISTRATOR,None,None,Perempuan,None,None,Belum Menikah,None,None,None,None,None,None,None,ditari@leapsurabaya.sch.id,None,None,None,None,None,None,None,None,None,None,None,2026-01-01,None,2,1,logo.png,None
1,2,U00003,LEAP00313III23,3514186411980002,"Graciela Evanda Ronadi, S.Kom.",Graciela,Sidoarjo,1998-11-24,Perempuan,None,Islam,Belum Menikah,"Ranggeh, Gondangwetan, Pasuruan","Lebak Indah Regency B40, Surabaya",WNI,1,None,Membaca,https://www.linkedin.com/in/graciela-evanda-ronadi-b134221a3/?originalSubdomain=id,graciela@leapsurabaya.sch.id,graciela@leapsurabaya.sch.id,0812-3447-7137,53.856.239.8-619.000,None,None,0140881385821,Sepeda Motor,https://www.instagram.com/gracielaevr/,None,https://drive.google.com/drive/folders/1alw9SubNFKSs40YOrz36JlZCGj7T8waC?usp=sharing,"Maag, tipes",2026-01-01,None,2,1,1707279559_ec864cc58f50e8b890d5.jpg,1702002184_d82747bcbd0f2bdab0a7.png
2,3,U00011,LEAP01101XII20,3515135105910001,DANIAR AULIA RIZKI,DANIAR,SURABAYA,1991-05-11,Perempuan,B,Islam,Menikah,"PONDOK TROSOBO INDAH L-5, RT 003 RW 009, TROSOBO, TAMAN, SIDOARJO, JAWA TIMUR, INDONESIA.",GUNUNGANYAR LOR GG 3A NO 9A,WNI,1,1,"MAIN GAME, NONTON",https://www.linkedin.com/in/daniar-aulia-rizki-28b8a4250/,daniar.rizki@leapsurabaya.sch.id,daniar.rizki@leapsurabaya.sch.id,0896-9632-0278,46.770.512.5-603.000,21034278636,0002319622514,1410022279715,Sepeda Motor,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va7B0PtK57XOZrl-1nvu1n3Wz1jOj75r,PREKLAMSIA,2026-01-01,None,2,0,1692700456_2d6352eb557d734e53b7.jpeg,None
3,4,U00012,LEAP01202III20,3578106705930001,Habibah Melyna,Habibah,Semarang,1993-05-27,Perempuan,AB,Islam,Belum Menikah,Jl Pacar Kembang Vc No 22,Jl Pacar Kembang Vc No 22,WNI,1,None,Mereview berbagai jenis makanan,https://www.linkedin.com/in/habibah-melyna/,habibah.elfiani@leapsurabaya.sch.id,habibah.elfiani@leapsurabaya.sch.id,0857-3093-3317,96.917.411.9-619.000,21034278651,0002999218792,1420018314442,Sepeda Motor,https://instagram.com/habibahmelyna?igshid=MzNlNGNkZWQ4Mg==,None,None,"Alergi udang, pengawet makanan dan micin",2026-01-01,desain grafis,3,1,1686127161_1ec3d11da554fb668e7c.jpg,1702892302_89507e3f2d87fb0dfdee.png
4,5,U00014,LEAP01431VII18,3578035706820005,Laksmi Puspitowardhani,Laksmi,Surabaya,1982-06-17,Perempuan,O,Islam,Belum Menikah,Rungkut Asri Barat XIII no 17,Rungkut Asri Barat XIII no 17,WNI,2,None,"Hiking, Reading, Traveling, Attending Concert",https://www.linkedin.com/in/laksmi-puspitowardhani-ab1a1336,laksmi.p@leapsurabaya.sch.id,laksmi.p@leapsurabaya.sch.id,6282-1399-6681,35.227.276.9-615.000,18088838356,00015384308,1420016807967,Sepeda Motor,https://www.instagram.com/laksmi_purplespace/?hl=en,None,https://drive.google.com/drive/folders/1-7iI4-phQIGBra_G-bekA_2Fuu_-nAiu?usp=drive_link,Liver,2026-01-01,None,3,1,1696338938_2a535db26e3f89919325.jpg,None
5,6,U00015,LEAP01514II11,None,Ika Asriani Yadin,Ika,None,None,Perempuan,None,None,Belum Menikah,None,None,None,None,None,None,None,ikayadin@leapsurabaya.sch.id,None,None,None,None,None,None,None,None,None,None,None,2026-01-01,None,3,1,1694397431_30b88a94a51ff7c8024e.png,None
6,7,U00016,LEAP01619VI17,3524035806960001,"Luluk Fatikah Sari, S.Pd.",Luluk,Lamongan,1996-06-18,Perempuan,None,Islam,Belum Menikah,"RT 001/RW 002, Ngangkrok, Medalem, Modo, Lamongan","Jalan Pabrik Kulit Gg. 2 Gg. Buntu No.24, Jemur Wonosari, Wonocolo, Surabaya",WNI,1,None,Menonton Drama/Film,https://www.linkedin.com/in/luluk-fatikah-sari-a74001252/,luluk.sa

Processed Data for keluarga_karyawan table:


,id_keluarga,id_karyawan,hubungan_keluarga,nama_lengkap,pekerjaan,nomor_hp
0,None,2,Ibu,Nur Fatmawati,Wiraswasta,081234477137
1,None,9,Suami,Fathul Kirom,Suami,0852-3101-7799
2,None,12,Ayah,"Suwandi, S.Pd.",Guru Matematika SMAN 17 Surabaya,087853591616
3,None,12,Ibu,"Ir. Hj. Erhasyati Islamiyah, M.M.",(Pensiun) Guru Biologi SMA Muhammadiyah 2 Surabaya,08179365966
4,None,3,Suami,JONATHAN O'DRISCOLL,TIDAK BEKERJA,089696320278
5,None,12,Adik,Fahada Akbariyah,Pekerja Laboratoriumm RS Soewandhi,088801762951
6,None,12,Adik,Rafi Arif Billah,Mahasiswa S1 Manajemen Undika,08563002290
7,None,18,Ayah,Herrie Susanto,Wiraswasta,08123429138
8,None,18,Ibu,Juliani,tidak bekerja,081911541719
9,None,17,Saudara,AGUS DJOKO PRANOWO,KARYAWAN SWASTA,087852999236


Processed Data for bidang_kategori table:


,id_bidang_kategori,nama_kategori_bidang,id_bidang
0,7,Brand Identity,7
1,13,Training,9
2,14,Referensi,9
3,16,Training,8
4,17,Training,11
5,18,Referensi,11
6,19,Referensi,8
7,20,Referensi,7
8,21,Training,7
9,22,Proposal,11


,idbidang,namabidang
0,7,Sales & Marketing
1,8,R & D
2,9,Sosial Media
3,11,Komunitas


Processed Data for bidang_link table:


,id_bidang_link,nama_form,link_drive,id_bidang_kategori,status_share
0,16,Daftar Training,https://drive.google.com/drive/folders/1BCPhp6eG5mNeWyTZS8i4xHEor-zgeyJu?usp=drive_link,13,0
1,17,Referensi,https://drive.google.com/drive/folders/1Lo6hzugBnuacIyHPF6nef-9BV1O9mnCQ?usp=drive_link,14,0
2,18,Dokumentasi,https://drive.google.com/drive/folders/14ieegP1eR_Vdj-QCOhevVeKR2DOaQ5_w?usp=drive_link,24,0
3,19,List Training,https://drive.google.com/drive/folders/1yivPYk3nJ3XmRZ5Np3S7d30AK3yw14GJ?usp=drive_link,16,0
4,20,Referensi,https://drive.google.com/drive/folders/1ah2G3XtafGmA3B1V01Np8rmv8lZrUmKl?usp=drive_link,19,0
5,21,Logo Leap,https://drive.google.com/drive/folders/1S1Og9cSimI2zuwCjD6PpBgSHqubhfcjI?usp=sharing,7,0
6,22,Referensi,https://drive.google.com/drive/folders/1JuWEPTbtE58G_1FzbLwZyjJhARpO7Pq-?usp=sharing,20,0


In [39]:
import json
import pickle
with open('fase_2_cimut.pkl', 'wb') as f:
    pickle.dump(df_new, f)

print("✓ Data df_new sudah disimpan ke df_new.pkl")
print("Siap untuk digunakan di insert_handler.ipynb")

✓ Data df_new sudah disimpan ke df_new.pkl
Siap untuk digunakan di insert_handler.ipynb
